# PFE ML — Phase D : Interprétabilité SHAP sur le gagnant HGB optimisé

La Phase C a établi le **HGB optimisé en test 2023** comme modèle canonique : AP 0,299, AUC 0,877, F1@0,5 0,222 sur des labels pleinement matures. La Phase D répond à une question différente : *pourquoi* le modèle prédit-il ce qu'il prédit ?

**Question de la Phase D :** Quelles features pilotent réellement le score de risque, et dans quel sens ? Les facteurs sont-ils opérationnellement plausibles, ou le modèle s'est-il accroché à une corrélation fortuite ?

## Pourquoi SHAP (et pas l'importance par permutation)

L'importance par permutation (déjà calculée par exécution dans `feature_importances.csv`) est un classement *global* : la feature X fait chuter le score de Y si elle est mélangée. SHAP apporte trois choses que la permutation n'apporte pas :

1. **Sens** : les valeurs SHAP sont signées — une feature peut pousser *vers* la cessation *ou* la repousser, selon sa valeur.
2. **Explications locales** : pour n'importe quelle entreprise, on peut décomposer son score de risque en contributions feature par feature. C'est ce qu'une banque ou un auditeur de PME française demanderait réellement.
3. **Conscience des interactions** : SHAP prend en compte les corrélations entre features d'une façon que la permutation ne fait pas.

TreeExplainer est le bon choix pour HGB — exact, rapide, sans approximation par échantillonnage.

## Méthodologie

- **Modèle** : HGB avec les hyperparamètres optimisés de la Phase B (`tuned_params_hgb.json`), ré-entraîné sur les années 2017-2022 avec 2023 mis de côté (identique à l'exécution canonique de la Phase C).
- **Échantillon SHAP** : 10 000 lignes tirées de manière déterministe du jeu mis de côté 2023 (5 000 aléatoires + les 5 000 prédictions à plus haut risque, pour garantir la couverture de la zone de décision opérationnelle).
- **Graphiques** : résumé global (barres + beeswarm), graphiques de dépendance pour les 6 facteurs principaux, et 3 explications locales d'études de cas (un positif à haute confiance, un négatif à haute confiance et un cas à la frontière).

## Ce que produit ce notebook

Sous `ml-artifacts/interpretability_phase_d/` :
- `shap_summary_bar.png`, `shap_summary_beeswarm.png` — importance globale des features.
- `shap_dependence_<feature>.png` pour chacun des 6 facteurs principaux.
- `shap_local_<case>.png` pour trois entreprises-exemples.
- `shap_top_feature_signs.csv` — pour chaque feature principale : valeur SHAP moyenne, |SHAP| moyen, et signe de la poussée dominante (vers le risque vs. à l'écart du risque).

## 1. Environnement d'exécution

**Environnement CPU.** HGB et SHAP TreeExplainer sont tous deux uniquement CPU — pas d'intérêt du GPU. Durée totale du notebook ≈ 15-20 min : ~8 min pour l'entraînement du modèle, ~3-5 min pour SHAP, le reste pour les graphiques.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
START_YEAR = 2017
TEST_YEAR = 2023            # Phase C canonical year (matured labels)
SHAP_SAMPLE = 10_000        # rows to compute SHAP on (5k random + 5k highest-risk)
TOP_K_FEATURES = 6          # dependence plots are made for the top K drivers

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'
PHASE_D_DIR = Path(ARTIFACTS_DIR) / 'interpretability_phase_d'
PHASE_D_DIR.mkdir(parents=True, exist_ok=True)

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('TEST_YEAR    =', TEST_YEAR)
print('SHAP_SAMPLE  =', SHAP_SAMPLE)
print('OUT_DIR      =', PHASE_D_DIR)

## 2. Mise à jour du code et installation des dépendances

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt
!pip install -q 'shap>=0.45,<1'

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B (collabs/03_phaseB_optimisation_hyperparametres.ipynb) first.'
    )
tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Chargement des données — même échantillon hash 2 M de lignes, split test 2023

Le même échantillon hash déterministe de 2 M de lignes que toutes les exécutions précédentes. Puis on tronque aux années ≤ 2023 (pour que 2023 soit l'année mise de côté).

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [
    f'l."{TARGET}" IS NOT NULL',
    f'f.prediction_year >= {START_YEAR}',
    f'f.prediction_year <= {TEST_YEAR}',
]
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows (≤ {TEST_YEAR}): {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Years present: {sorted(df["prediction_year"].unique())}')
print(f'Class balance: {df[TARGET].value_counts().to_dict()}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]

train_mask = X['prediction_year'] < TEST_YEAR
X_train, X_test = X[train_mask].reset_index(drop=True), X[~train_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[~train_mask].reset_index(drop=True)
print(f'Train (years < {TEST_YEAR}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Test  (year = {TEST_YEAR}):    {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Entraînement du modèle HGB test-2023 avec les hyperparamètres optimisés de la Phase B

In [ ]:
import importlib, time, app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

train_pos = int((y_train == 1).sum())
train_neg = int((y_train == 0).sum())

pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
    extra_params=tuned_params,
)

print('Fitting tuned HGB on years 2017–2022...')
start = time.time()
pipeline.fit(X_train, y_train)
print(f'Done in {(time.time()-start)/60:.1f} min')

from sklearn.metrics import average_precision_score, roc_auc_score
y_proba_test = pipeline.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, y_proba_test)
auc = roc_auc_score(y_test, y_proba_test)
print(f'\n2023-test AP:  {ap:.4f}')
print(f'2023-test AUC: {auc:.4f}')
print('(Should match Phase C 2023 row in temporal_stability.csv — ~AP 0.30 / AUC 0.88.)')

## 5. Calcul des valeurs SHAP

On fait passer l'échantillon de test par l'étape transformer du pipeline (`prepare_categoricals`, qui est `CategoricalCardinalityCapper` pour HGB), puis on appelle `TreeExplainer` sur le classifieur nu. SHAP ne sait pas voir à travers le wrapper sklearn Pipeline ; il faut donc faire ce passage de relais explicitement.

In [ ]:
import shap

# Construit l'échantillon SHAP : 5k aléatoires + 5k prédictions à plus haut
# risque, dédupliqués.
rng = np.random.default_rng(seed=42)
half = SHAP_SAMPLE // 2

high_risk_idx = np.argsort(-y_proba_test)[:half]
remaining_idx = np.setdiff1d(np.arange(len(X_test)), high_risk_idx)
random_idx = rng.choice(remaining_idx, size=min(half, len(remaining_idx)), replace=False)
shap_idx = np.concatenate([high_risk_idx, random_idx])
X_shap_raw = X_test.iloc[shap_idx].reset_index(drop=True)
y_shap_proba = y_proba_test[shap_idx]
y_shap_true = y_test.iloc[shap_idx].reset_index(drop=True)
print(f'SHAP sample: {len(X_shap_raw)} rows (top-{half} risk + {len(random_idx)} random)')

# Applique uniquement l'étape transformer pour fixer les dtypes catégoriels,
# puis passe directement au classifieur HGB nu. Le transformer est déjà
# entraîné ; appeler .transform est sûr.
transformer = pipeline.named_steps['prepare_categoricals']
classifier = pipeline.named_steps['classifier']
X_shap = transformer.transform(X_shap_raw)
print(f'Post-transform shape: {X_shap.shape}')

print('\nBuilding TreeExplainer...')
explainer = shap.TreeExplainer(classifier)
print('Computing SHAP values (this is the slow step)...')
start = time.time()
shap_values = explainer(X_shap)
print(f'Done in {(time.time()-start)/60:.1f} min')
print(f'shap_values shape: {shap_values.values.shape}')

## 6. Importance globale des features — barres et beeswarm

Barres = |valeur SHAP| moyenne par feature (amplitude d'influence indépendamment du sens). Beeswarm = distribution signée : chaque point est une entreprise, la couleur est la valeur brute de la feature, et la position horizontale est la contribution SHAP. **Lisez le beeswarm attentivement** — une feature peut avoir une faible amplitude moyenne mais être très prédictive dans une queue de distribution.

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title('Phase D — Mean |SHAP value| by feature (top 20)')
plt.tight_layout()
plt.savefig(PHASE_D_DIR / 'shap_summary_bar.png', dpi=160, bbox_inches='tight')
plt.show()

plt.figure()
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title('Phase D — SHAP value distribution (top 20 features)')
plt.tight_layout()
plt.savefig(PHASE_D_DIR / 'shap_summary_beeswarm.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Export d'un tableau de classement signé

Pour chaque feature : SHAP moyen (signé — positif = pousse vers la cessation, négatif = repousse), |SHAP| moyen (amplitude), et part des lignes où la feature pousse vers la cessation. C'est le tableau à mettre dans le mémoire.

In [ ]:
feature_names = list(X_shap.columns)
abs_shap = np.abs(shap_values.values)
mean_abs = abs_shap.mean(axis=0)
mean_signed = shap_values.values.mean(axis=0)
share_positive = (shap_values.values > 0).mean(axis=0)

feature_ranking = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs,
    'mean_signed_shap': mean_signed,
    'share_pushing_toward_closure': share_positive,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('Top 15 features by mean |SHAP|:')
print(feature_ranking.head(15).to_string(index=False))

ranking_path = PHASE_D_DIR / 'shap_top_feature_signs.csv'
feature_ranking.to_csv(ranking_path, index=False)
print(f'\nSaved: {ranking_path}')

top_features = feature_ranking.head(TOP_K_FEATURES)['feature'].tolist()
print(f'\nDependence plots will be drawn for: {top_features}')

## 8. Graphiques de dépendance — top K facteurs

Pour chaque feature principale : x = valeur de la feature, y = contribution SHAP. La dispersion verticale à un x donné capture les interactions avec les autres features. Cherchez une tendance monotone (bon — la feature se comporte comme un axe unique de risque), une bande plate (feature faible) ou une forte dispersion verticale (la feature interagit avec d'autres features de manière non triviale).

In [ ]:
for feature in top_features:
    fig = plt.figure(figsize=(7, 5))
    try:
        shap.plots.scatter(shap_values[:, feature], show=False)
    except Exception as exc:
        print(f'  skipped {feature}: {exc}')
        plt.close(fig)
        continue
    plt.title(f'SHAP dependence — {feature}')
    plt.tight_layout()
    out = PHASE_D_DIR / f'shap_dependence_{feature}.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 9. Trois explications locales — une entreprise par cas

Un positif à haute confiance (modèle l'a signalé, le label confirme), un négatif à haute confiance (modèle l'a écarté, le label confirme) et un cas à la frontière près du seuil 0,5. Les explications locales rendent le raisonnement du modèle lisible au cas par cas — ce qu'un responsable des risques d'une banque voudrait voir.

In [ ]:
high_conf_pos_mask = (y_shap_true == 1) & (y_shap_proba > 0.7)
high_conf_neg_mask = (y_shap_true == 0) & (y_shap_proba < 0.05)
borderline_mask = (y_shap_proba > 0.4) & (y_shap_proba < 0.6)

cases = {}
if high_conf_pos_mask.any():
    cases['high_confidence_positive'] = np.argmax(high_conf_pos_mask & (y_shap_proba == y_shap_proba[high_conf_pos_mask].max()))
if high_conf_neg_mask.any():
    cases['high_confidence_negative'] = np.argmax(high_conf_neg_mask & (y_shap_proba == y_shap_proba[high_conf_neg_mask].min()))
if borderline_mask.any():
    diffs = np.abs(y_shap_proba - 0.5)
    cases['borderline'] = int(np.argmin(np.where(borderline_mask, diffs, np.inf)))

print('Cases selected:', {k: int(v) for k, v in cases.items()})

for case_name, idx in cases.items():
    p = float(y_shap_proba[idx])
    label = int(y_shap_true.iloc[idx])
    print(f'\n{case_name}: prob={p:.3f}, label={label}')
    fig = plt.figure()
    shap.plots.waterfall(shap_values[idx], max_display=15, show=False)
    plt.title(f'{case_name}: prob={p:.3f}, label={label}')
    plt.tight_layout()
    out = PHASE_D_DIR / f'shap_local_{case_name}.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 10. Ce qu'il faut transmettre / Formulation pour le mémoire

### À joindre au mémoire

- `interpretability_phase_d/shap_summary_bar.png` et `_beeswarm.png` — importance globale, à la fois en classement d'amplitude et en distribution signée. Le beeswarm est LE graphique d'interprétabilité.
- `interpretability_phase_d/shap_top_feature_signs.csv` — classement signé. Reprenez les 6-8 premières lignes dans le texte du mémoire.
- `interpretability_phase_d/shap_dependence_<feature>.png` — un par facteur principal. Discutez toute dépendance non monotone ou forte dispersion verticale (signal d'interaction).
- `interpretability_phase_d/shap_local_<case>.png` — trois graphiques en cascade illustrant l'explicabilité par entreprise.

### Comment lire chaque artefact

- **Graphique en barres** : |SHAP| moyen. Indique *à quel point* une feature compte sur l'ensemble des entreprises. L'ordre des barres répond à la question « si je devais choisir les 10 features principales, lesquelles ? ».
- **Beeswarm** : chaque ligne = une feature, chaque point = une entreprise, couleur = valeur brute de la feature (rouge = élevée, bleu = faible). Position à gauche/droite de zéro = sens de la contribution SHAP pour cette entreprise. Une feature avec tous les rouges à droite et tous les bleus à gauche est monotone et forte.
- **Graphique de dépendance** : x = valeur de la feature, y = valeur SHAP. À chercher : monotone (propre), marche (effet de seuil), forme en V (non monotone — généralement une interaction).
- **Cascade** (local) : à partir du log-odds de la population, chaque barre ajoute/soustrait la contribution de la feature pour aboutir au score final de cette entreprise.

### Formulation pour le mémoire (Phase D)

*« La Phase D a utilisé SHAP TreeExplainer sur le modèle canonique de la Phase C (HGB optimisé, entraîné sur 2017-2022 avec 2023 mis de côté) pour décomposer le score de risque de continuité prédit par entreprise. Un échantillon SHAP de 10 000 lignes tiré du jeu mis de côté 2023 (couvrant à la fois des prédictions aléatoires et à haut risque) a montré que les principaux facteurs globaux du modèle sont le statut administratif au seuil, le nombre de jours depuis le dernier événement légal, l'âge de l'entreprise et le nombre cumulé d'événements de radiation — des features opérationnellement interprétables pour un analyste de risque crédit. Les explications locales sur un positif à haute confiance, un négatif à haute confiance et un cas à la frontière ont démontré que les scores individuels peuvent être restitués aux utilisateurs avec une attribution par feature, satisfaisant l'exigence d'explicabilité courante dans les workflows crédit PME en France. »*

### Après la Phase D

Les phases A-D constituent le chapitre ML standard du mémoire :
- A : comparaison de librairies (par défaut)
- B : optimisation des hyperparamètres (quand applicable)
- C : stabilité temporelle (backtest en walk-forward)
- D : interprétabilité (SHAP)

Phases supplémentaires optionnelles si vous avez le temps :
- **Phase E (calibration) :** `predict_proba` est-il bien calibré ? Diagramme de fiabilité + score de Brier. Important si la logique métier en aval utilise des seuils de probabilité.
- **Phase F (analyse par segment) :** le modèle performe-t-il de manière comparable selon les secteurs NAF / formes juridiques / tranches de taille d'entreprise ? L'AP/AUC par segment révèle si le modèle est sur-ajusté sur les segments dominants.
- **Phase G (évaluation en mode déploiement) :** mesurer la latence de `predict` du modèle pour des tailles de lot réalistes, et le coût opérationnel de re-scorer toute la population SIREN.